# 第1章 ローカルLLMの基本操作 実践

この Notebook は、文書 `01-overview.md` に対応する最初の体験編です。文書01はローカルLLMの全体像を説明しますが、この Notebook ではまず「開く」「話しかける」「返答を見る」までを体験します。

この Notebook では、Ollama API に接続する教材用の小さなブラウザ Chat UI を起動します。追加アプリをインストールしなくても、ブラウザでチャットの体験ができます。

この章で行うこと:

1. Ollama が動いていることを確認する
2. Python からLLMを呼び出す
3. ブラウザ Chat UI を起動して会話する
4. Agentic AI として aider を起動し、最初の安全な操作を試す

In [ ]:
from pathlib import Path
import os
import sys

# Notebook をどこから開いても helper を import できるようにします。
search_roots = [Path.cwd()]
env_root = os.environ.get("LOCAL_LLM_REPO_ROOT")
if env_root:
    search_roots.append(Path(env_root))
search_roots.append(Path("C:/LLM"))

seen = set()
for root in search_roots:
    current = root.resolve()
    for candidate in [current, *current.parents]:
        if candidate in seen:
            continue
        seen.add(candidate)
        helper_dir = candidate / "notebooks"
        if (helper_dir / "local_llm_practice.py").exists():
            sys.path.insert(0, str(helper_dir))
            break
    else:
        continue
    break
else:
    raise RuntimeError(
        "notebooks/local_llm_practice.py が見つかりません。"
        "C:/LLM か notebooks/ 配下で開くか、LOCAL_LLM_REPO_ROOT を設定してください。"
    )

from local_llm_practice import (
    check_ollama,
    configure_local_caches,
    copy_to_clipboard,
    load_chapter,
    ollama_generate,
    open_aider_terminal,
    prepare_aider_practice_workspace,
    print_headings,
    REPO_ROOT,
    DOCS_DIR,
    WORK_DIR,
    start_ollama_chat_ui,
    stop_ollama_chat_ui,
)

configure_local_caches()
print("REPO_ROOT:", REPO_ROOT)
print("DOCS_DIR :", DOCS_DIR)
print("WORK_DIR :", WORK_DIR)

chapter_path, chapter_text = load_chapter("01-overview.md")
print(chapter_path)
print_headings(chapter_text)


## 1. Ollama が使えるか確認する

次のセルは `ollama --version` と `ollama list` を実行します。モデルが1つ以上表示されれば、この後の Python 呼び出しと Chat UI へ進めます。

In [ ]:
check_ollama()

## 2. Python からLLMへ話しかける

ここは Notebook が得意な部分です。セルを実行すると、Ollama API 経由でローカルLLMに質問します。

In [ ]:
response = ollama_generate(
    "ローカルLLMを文書作成や資料確認で使う時、最初に試す小さな作業を3つ提案してください。",
    temperature=0.2,
)
print(response)

## 3. ブラウザ Chat UI を開く

次のセルを実行すると、Notebook の裏側で小さなWebサーバーが起動し、Ollama に接続する Chat UI が開きます。

ブラウザが自動で開かない場合は、セル出力に表示される `http://127.0.0.1:...` のURLをブラウザへ貼り付けてください。

開いた画面で次を体験します。

1. 入力欄に質問を書く
2. `送信` を押す
3. 返答が出るのを待つ
4. 追加で「もっと短く」「初めて読む人向けに」と頼む

In [ ]:
chat_prompt = "ローカルLLMを文書作成や資料確認で使う時、最初に試す小さな作業を3つ提案してください。"
start_ollama_chat_ui()
copy_to_clipboard(chat_prompt)


## 4. Agentic AI を起動する

ここでは、エージェンティックAIの最初の体験として `aider` を起動します。次のセルを実行すると別の PowerShell が開きます。

開いた PowerShell でまず次を試します。

1. `/help` と入力して、基本操作を見る
2. `/add README.md` と入力して、README を作業対象に追加する
3. `READMEの内容を初心者向けに要約してください。まだ編集しないでください。` と入力する
4. 返答を読んだら、編集はせず `/exit` で終了する

ここでは「ファイルを読ませる」「まだ編集しないように指示する」「終了する」までが体験です。
このセルは別の PowerShell を開きます。Notebook 側のファイルは編集しませんが、aider で編集依頼をすると差分が作られるため、この章では必ず `まだ編集しないでください` と伝え、最後は `/exit` で終了します。

実リポジトリを誤って編集しないように、この章では `work/aider-practice/` に作る練習用 workspace を開きます。ここは `.gitignore` されているため、教材の tracked files は変更されません。


In [ ]:
practice_path = prepare_aider_practice_workspace()
open_aider_terminal(practice_path)


## 結果の読み方 / 次へ進む判断

- `ollama list` にモデルが表示され、Python API のセルで返答か分かりやすいエラーが出れば、呼び出し経路は確認できています。
- Chat UI で質問を貼り付けて返答を見られたら、会話しながら条件を足す入口を体験できています。
- aider では `/add` したファイルの内容を説明できれば、ファイル文脈を持たせた相談の入口を確認できています。編集はまだしません。

ここまで進めば、第2章で同じ依頼をプロンプト設計として整える準備ができています。

Chat UI を使い終わったら、同じ kernel で `stop_ollama_chat_ui()` を実行すると教材用サーバーを終了できます。Notebook の kernel を再起動しても終了します。
